In [1]:
import warnings
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import  BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
docs = [
    "Q4 revenue for Project Alpha: income pushbacks delayed booking.",
    "IT ticket #4471: Project Alpha laptop revenue dashboard down.",
    "Financial report - Q4 Alpha income recognized this quarter.",
]

In [4]:
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

faiss_store = FAISS.from_texts(docs, embedder)

faiss_retriever = faiss_store.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6163.34it/s]


In [5]:
bm25_retriever = BM25Retriever.from_texts(docs)
bm25_retriever.k = 3

In [6]:
hybrid = EnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)

In [7]:
query = "04 Alpha revenue"

In [8]:
for data in faiss_retriever.invoke(query):
    print("-", data.page_content)

- Financial report - Q4 Alpha income recognized this quarter.
- Q4 revenue for Project Alpha: income pushbacks delayed booking.
- IT ticket #4471: Project Alpha laptop revenue dashboard down.


In [9]:
for data in bm25_retriever.invoke(query):
    print("-", data.page_content)

- IT ticket #4471: Project Alpha laptop revenue dashboard down.
- Financial report - Q4 Alpha income recognized this quarter.
- Q4 revenue for Project Alpha: income pushbacks delayed booking.


In [10]:
for data in hybrid.invoke(query):
    print("-", data.page_content)

- Financial report - Q4 Alpha income recognized this quarter.
- IT ticket #4471: Project Alpha laptop revenue dashboard down.
- Q4 revenue for Project Alpha: income pushbacks delayed booking.


In [11]:
from sentence_transformers import CrossEncoder

In [12]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16551.49it/s]


In [13]:
candidates = [data.page_content for data in hybrid.invoke(query)]

pairs = [(query, candidate) for candidate in candidates]
score = cross_encoder.predict(pairs)

reranked = sorted(zip(candidates, score), key= lambda  x: -x[1])

In [14]:
for candidate in candidates:
    print("-", candidate)

- Financial report - Q4 Alpha income recognized this quarter.
- IT ticket #4471: Project Alpha laptop revenue dashboard down.
- Q4 revenue for Project Alpha: income pushbacks delayed booking.


In [15]:
for candidate, score in reranked:
    print(f"{candidate}: {score}")

Q4 revenue for Project Alpha: income pushbacks delayed booking.: 0.23138269782066345
IT ticket #4471: Project Alpha laptop revenue dashboard down.: -2.1504290103912354
Financial report - Q4 Alpha income recognized this quarter.: -3.302947998046875
